In [2]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)

num_cols = [
    'HAARBRADES', 'HANOARBRAD', 'SuperficieTotalForestal',
    'HANOFOREST', 'SuperficieAgricola',
    'OtrasSuperficiesNoforestales', 'SuperficieTotalNoForestal'
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['SuperficieTotal'] = (
    df['SuperficieTotalForestal'].fillna(0)
    + df['SuperficieTotalNoForestal'].fillna(0)
)


def any_des_de_data(serie):
    dates = pd.to_datetime(serie, dayfirst=True, errors='coerce', format='mixed')
    return dates.dt.year


any_1 = any_des_de_data(df['Data incendi'])
any_2 = any_des_de_data(df['Data incendi'])
any_campania = df['Campania'].astype(str).str.extract(r'(\d{4})\s*$')[0].astype(float)
df['Any'] = any_1.combine_first(any_2).combine_first(any_campania)

df = df.dropna(subset=['Nom comarca', 'Any']).copy()
df['Any'] = df['Any'].astype(int)
df = df[(df['Any'] >= 1998) & (df['Any'] <= 2024)]

# Unifiquem causes buides o "Desconeguda" amb la categoria "Desconocida" real del CSV
_causes_valides = df['Causa'].dropna().astype(str).str.strip()
_candidata_desconeguda = next(
    (c for c in _causes_valides.unique() if 'esconoc' in c.lower()),
    next((c for c in _causes_valides.unique() if 'esconeg' in c.lower()), 'Desconocida')
)
df['Causa'] = df['Causa'].fillna(_candidata_desconeguda).astype(str).str.strip()
df['Causa'] = df['Causa'].replace({
    'nan': _candidata_desconeguda,
    '': _candidata_desconeguda,
    'Desconeguda': _candidata_desconeguda,
    'desconeguda': _candidata_desconeguda
})

for c in ['Nom comarca', 'Nom vegueria']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

correccions_comarca = {
    'Aran': "Val d'Aran",
    'Montisà': 'Montsià',
    "Val d'Aran": "Val d'Aran"
}
df['Nom comarca'] = df['Nom comarca'].replace(correccions_comarca)

top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )


# %% ===================================================================
# Q1 · v1 (2 Mapes: Hectàries i Nombre d'incendis) + v2 (1 Mapa amb distorsió)
# =======================================================================
df_comarques = df.groupby('Nom comarca', as_index=False).agg(
    Superficie_Total_Calcinada=('SuperficieTotal', 'sum'),
    Nombre_Incendis=('SuperficieTotal', 'count'),
    Mida_Mitjana=('SuperficieTotal', 'mean')
).round(2)

df_comarques_complet = pd.DataFrame({'Nom comarca': sorted(list(noms_geojson))}).merge(
    df_comarques, on='Nom comarca', how='left'
).fillna({'Superficie_Total_Calcinada': 0, 'Nombre_Incendis': 0, 'Mida_Mitjana': 0})

w1, h1 = mida_mapa(540)

mapa_q1_inc = alt.Chart(comarques_geo).mark_geoshape(
    stroke='white', strokeWidth=0.9
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Nombre_Incendis:Q',
        scale=alt.Scale(scheme='oranges'),
        title="Nombre d'incendis",
        legend=alt.Legend(orient='bottom', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis", format=','),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Superfície calcinada (ha)', format=',.1f')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
    width=w1, height=h1,
    title=alt.TitleParams("Freqüència: Nombre total d'incendis per comarca", fontSize=15, fontWeight='bold')
)

mapa_q1_ha = alt.Chart(comarques_geo).mark_geoshape(
    stroke='white', strokeWidth=0.9
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Superficie_Total_Calcinada:Q',
        scale=alt.Scale(scheme='reds', type='symlog', constant=50),
        title='Hectàrees calcinades (ha)',
        legend=alt.Legend(orient='bottom', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Superfície calcinada (ha)', format=',.1f'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis", format=',')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
    width=w1, height=h1,
    title=alt.TitleParams("Impacte: Hectàrees totals calcinades per comarca", fontSize=15, fontWeight='bold')
)

q1_v1 = alt.hconcat(mapa_q1_inc, mapa_q1_ha, spacing=24).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q1 (v1) · Freqüència vs. Impacte: on s'encenen més focs no sempre és on es crema més bosc",
        "Comparativa per comarca del nombre total d'incendis (esquerra) i les hectàrees calcinades (dreta) a Catalunya (1998-2024)"
    )
).configure_view(stroke=None)
q1_v1
q1_v1.save('outputs/Q1V1.json')


# --- Q1 v2: 1 sol mapa amb DISTORSIÓ (Cartograma no-contigu segons Nombre d'Incendis + Color = Hectàrees) ---
def crear_cartograma_distorsionat(geo_d, df_stats):
    """
    Deforma/escala el polígon de cada comarca respecte al seu centroide
    en funció del nombre d'incendis (Cartograma d'escala de polígons),
    mantenint la silueta de Catalunya reconeixible.
    """
    geo_copy = copy.deepcopy(geo_d)
    max_inc = max(df_stats['Nombre_Incendis'].max(), 1)
    inc_map = df_stats.set_index('Nom comarca')['Nombre_Incendis'].to_dict()

    for feat in geo_copy['features']:
        nom = feat['properties']['NOMCOMAR']
        n_inc = inc_map.get(nom, 0)
        # Factor d'escala lineal amb arrel quadrada (perquè l'àrea sigui proporcional al nombre d'incendis)
        factor = 0.30 + 0.85 * np.sqrt(n_inc / max_inc)

        pts_x, pts_y = [], []

        def recolhir(coords):
            if isinstance(coords[0], (int, float)):
                pts_x.append(coords[0])
                pts_y.append(coords[1])
            else:
                for c in coords:
                    recolhir(c)

        recolhir(feat['geometry']['coordinates'])
        cx, cy = np.mean(pts_x), np.mean(pts_y)

        def escalar(coords):
            if isinstance(coords[0], (int, float)):
                return [cx + (coords[0] - cx) * factor, cy + (coords[1] - cy) * factor]
            return [escalar(c) for c in coords]

        feat['geometry']['coordinates'] = escalar(feat['geometry']['coordinates'])
    return geo_copy


geo_distorsionat = crear_cartograma_distorsionat(geo_dict, df_comarques_complet)
comarques_cartograma = alt.Data(values=geo_distorsionat['features'])
w1_carto, h1_carto = mida_mapa(720)

# Silueta de fons suau (mida real) + Comarques distorsionades segons nombre d'incendis i acolorides per hectàrees
fons_silueta = alt.Chart(comarques_geo).mark_geoshape(
    fill='#f2f2f2', stroke='#d0d0d0', strokeWidth=0.6
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y)

capa_distorsio = alt.Chart(comarques_cartograma).mark_geoshape(
    stroke='#222222', strokeWidth=0.9, fillOpacity=0.92
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Superficie_Total_Calcinada:Q',
        scale=alt.Scale(scheme='reds', type='symlog', constant=50),
        title='Superfície cremada (ha)',
        legend=alt.Legend(orient='right', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis (mida polígon)", format=','),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Hectàrees cremades (color)', format=',.1f'),
        alt.Tooltip('Mida_Mitjana:Q', title='Mida mitjana per foc (ha)', format=',.2f')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y)

q1_v2 = (fons_silueta + capa_distorsio).properties(
    width=w1_carto, height=h1_carto,
    title=titol_rt(
        "Q1 (v2) · Cartograma de distorsió: la mida de cada comarca reflecteix el nombre d'incendis i el color les hectàrees",
        "Les comarques amb més incendis s'expandeixen i les que en tenen pocs s'encongeixen sobre la silueta real (1998-2024)"
    )
).configure_view(stroke=None)

q1_v2.save('outputs/Q1V2.json')


FileNotFoundError: [Errno 2] No such file or directory: 'data/incendis_final.csv'